In [ ]:
import numpy as np
import cvxpy as cp
import pandas as pd
from pathlib import Path

In [ ]:
cp.MAX_NUM_SUBEXPRESSIONS = 10000

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data/combined_data_positive_gen.csv"
df = pd.read_csv(DATA_DIR, 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  
                     index_col='ts')

n = df["Collective PV"].size

# fix power/energy
# fix battery capacity = max(SOC)

In [ ]:
Pload_np = np.zeros((n))+2
Pload = cp.Constant(Pload_np)
minimum_self_sufficiency = 80

charge_eff = 0.95
discharge_eff = 0.95


battery_capacity = cp.Variable(n)

Pgen_pd = df["Collective PV"]+df["Aircon_WT Power"]
Pgen = cp.Constant(Pgen_pd)


Pcharge = cp.Variable(n,nonneg=True) # charge is positive
Pdischarge = cp.Variable(n,nonneg=True) # discharge is positive

Pgrid_buy = cp.Variable(n, nonneg=True)
Pgrid_sell = cp.Variable(n, nonneg=True)


SOC = cp.Variable(n)
self_sufficiency = (Pgen - Pcharge-Pgrid_sell)/Pload 
average_self_sufficiency = sum(self_sufficiency)/n

In [ ]:
constraints = [Pgen + Pdischarge - Pcharge + Pgrid_buy - Pgrid_sell == Pload, # power flow equation 
               SOC[1:] == SOC[:-1] + Pcharge[1:]*charge_eff - Pdischarge[1:]*discharge_eff, # battery state definition
               SOC >= 0.1 * battery_capacity, # minimum 10% discharge
               average_self_sufficiency >= minimum_self_sufficiency, # average self sufficiency over the year
               SOC[0] == 0.5 * battery_capacity+Pcharge[0]*charge_eff - Pdischarge[0]*discharge_eff] ## Starting with 50% charge
               

In [ ]:
for column in df.columns:
    print(column)

In [ ]:
problem = cp.Problem(cp.Minimize(cp.max(SOC)), constraints)
problem.solve()


In [ ]:
load = 1 # kW
battery_capacity = cp.variable(df.size)


# Construct the problem.
x = cp.Variable(n)
objective = cp.Minimize(cp.sum_squares(A @ x - b))
#constraints = [0 <= x, x <= 1]
prob = cp.Problem(objective, constraints)

# The optimal objective value is returned by `prob.solve()`.
result = prob.solve()
# The optimal value for x is stored in `x.value`.
print(x.value)
# The optimal Lagrange multiplier for a constraint is stored in
# `constraint.dual_value`.
print(constraints[0].dual_value)